# Gold Price Direction Prediction | توقع اتجاه الذهب

**الهدف:** التنبؤ بارتفاع (`1`) أو عدم ارتفاع (`0`) سعر إغلاق عقود الذهب الآجلة `GC=F` في **يوم التداول التالي**.

> مشروع تعليمي وليس توصية استثمارية. التوقع مرتبط بآخر جلسة تداول متاحة وليس بالضرورة غدًا بالتقويم. شغّلي الخلايا بالترتيب عبر **Runtime → Run all**.


## 1 — تثبيت المكتبات

In [ ]:
!pip -q install yfinance xgboost joblib

## 2 — المكتبات والإعدادات

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import joblib
from pathlib import Path
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from xgboost import XGBClassifier

OUT = Path('/content/gold_project')
OUT.mkdir(exist_ok=True)
FEATURES = ['Return_1d', 'Return_3d', 'MA7_Ratio', 'MA30_Ratio', 'Volatility_7d', 'Range_Pct', 'Volume_Change']
print('Output folder:', OUT)

## 3 — تنزيل الداتا الحقيقية وحفظ نسخة CSV

إذا تعذر الاتصال بمصدر الأسعار، أعيدي تشغيل الخلية لاحقًا. هذا الدفتر لا يستخدم بيانات اصطناعية بديلة.

In [ ]:
raw = yf.download('GC=F', start='2015-01-01', interval='1d', auto_adjust=True, progress=False)
if raw.empty:
    raise RuntimeError('لم يتم تنزيل البيانات. تحققي من اتصال Colab ثم أعيدي التشغيل.')
# دعم تنسيقات yfinance ذات الأعمدة متعددة المستويات
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw = raw.loc[:, ~raw.columns.duplicated()].copy()
required = ['Open', 'High', 'Low', 'Close', 'Volume']
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}; actual columns: {raw.columns.tolist()}')
raw = raw[required].copy()
raw.index = pd.to_datetime(raw.index).tz_localize(None) if raw.index.tz is not None else pd.to_datetime(raw.index)
raw.index.name = 'Date'
raw = raw.sort_index()
for col in required:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')
raw = raw.replace([np.inf, -np.inf], np.nan).dropna(subset=required)
raw = raw[~raw.index.duplicated(keep='last')]
if len(raw) < 200:
    raise ValueError('Not enough rows to train and evaluate reliably.')
raw.to_csv(OUT / 'gold_prices_raw.csv')
print('Rows:', len(raw), '| Date range:', raw.index.min().date(), 'to', raw.index.max().date())
display(raw.head())

## 4 — استكشاف الأسعار

In [ ]:
ax = raw['Close'].plot(figsize=(12,4), title='Gold futures — historical closing price')
ax.set_ylabel('USD per troy ounce')
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

## 5 — Feature Engineering + Target

كل الخصائص تستخدم بيانات الجلسة الحالية أو السابقة فقط. آخر صف بلا Target معروف، ونحتفظ به للتنبؤ القادم.

In [ ]:
df = raw.copy()
df['Return_1d'] = df['Close'].pct_change()
df['Return_3d'] = df['Close'].pct_change(3)
df['MA7_Ratio'] = df['Close'] / df['Close'].rolling(7).mean() - 1
df['MA30_Ratio'] = df['Close'] / df['Close'].rolling(30).mean() - 1
df['Volatility_7d'] = df['Return_1d'].rolling(7).std()
df['Range_Pct'] = (df['High'] - df['Low']) / df['Close']
df['Volume_Change'] = df['Volume'].pct_change()
df[FEATURES] = df[FEATURES].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=FEATURES).copy()
df['Next_Close'] = df['Close'].shift(-1)
df['Target'] = np.where(df['Next_Close'].notna(), (df['Next_Close'] > df['Close']).astype(int), np.nan)
latest_data = df[FEATURES].iloc[[-1]].copy()
data = df.dropna(subset=['Next_Close']).copy()
data['Target'] = data['Target'].astype(int)
if len(data) < 100:
    raise ValueError('Too few cleaned rows.')
data.to_csv(OUT / 'gold_cleaned.csv')
print('Cleaned rows:', len(data))
print('Target counts:')
print(data['Target'].value_counts())
display(data.tail())

## 6 — تقسيم زمني Train / Test (80% / 20%)

In [ ]:
split = int(len(data) * 0.8)
train, test = data.iloc[:split].copy(), data.iloc[split:].copy()
X_train, y_train = train[FEATURES], train['Target']
X_test, y_test = test[FEATURES], test['Target']
print('Train:', len(train), train.index.min().date(), '→', train.index.max().date())
print('Test: ', len(test), test.index.min().date(), '→', test.index.max().date())

## 7 — تدريب XGBoost

In [ ]:
model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                      subsample=0.9, colsample_bytree=0.9,
                      random_state=42, eval_metric='logloss', n_jobs=2)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Training complete')

## 8 — التقييم ومقارنة Baseline

Baseline يتوقع أن اتجاه الغد يساوي اتجاه عائد اليوم. دقة مرتفعة وحدها لا تعني نجاحًا في التداول.

In [ ]:
baseline_pred = (X_test['Return_1d'] > 0).astype(int)
results = pd.DataFrame({
    'Model': ['Baseline (same as today)', 'XGBoost'],
    'Accuracy': [accuracy_score(y_test, baseline_pred), accuracy_score(y_test, y_pred)],
    'F1_UP': [f1_score(y_test, baseline_pred, zero_division=0), f1_score(y_test, y_pred, zero_division=0)]
})
display(results.round(3))
print(classification_report(y_test, y_pred, target_names=['No Increase','UP'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No Increase','UP'], cmap='Blues')
plt.title('XGBoost — Test confusion matrix')
plt.show()

## 9 — أهمية الخصائص

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importance.plot(kind='barh', figsize=(8,4), title='Feature importance (XGBoost)')
plt.tight_layout()
plt.show()

## 10 — توقع يوم التداول القادم

هذا تقدير نموذجي غير معاير بالضرورة، ولا يمثل احتمالًا موثوقًا لتحقيق ربح.

In [ ]:
pred = int(model.predict(latest_data)[0])
prob_up = float(model.predict_proba(latest_data)[0, 1])
print('Last observed trading date:', df.index[-1].date())
print('Last close (USD/oz):', round(float(df['Close'].iloc[-1]), 2))
print('Next trading session direction:', 'UP' if pred == 1 else 'DOWN / NO INCREASE')
print('Model score for UP:', f'{prob_up:.1%}')

## 11 — حفظ النتائج والنموذج وتنزيل ZIP

## ملاحظات منهجية

- أسعار `GC=F` تمثل **عقود الذهب الآجلة** بالدولار للأونصة، وليست أسعار الذهب المحلية للجرام.
- هذا النموذج يتنبأ بالاتجاه بعد إغلاق آخر جلسة متاحة، ولا يعرف الأخبار أو الأحداث المستقبلية.
- عند تحديث النموذج أو ضبط معاملاته، استخدمي تحققًا زمنيًا `TimeSeriesSplit` مع إبقاء مجموعة الاختبار النهائية منفصلة.
- النتائج التاريخية لا تضمن أداءً مستقبلًا. لا يُستخدم النموذج وحده لاتخاذ قرارات استثمارية.


## 11 — Advanced Feature Engineering (without future data)
إضافة مؤشرات مبنية على البيانات الحالية والماضية فقط. نحتفظ بمجموعة الاختبار الزمنية النهائية بدون تغيير.

In [ ]:
advanced = df.copy()
advanced['Return_7d'] = advanced['Close'].pct_change(7)
advanced['MA14_Ratio'] = advanced['Close'] / advanced['Close'].rolling(14).mean() - 1
advanced['Momentum_5d'] = advanced['Close'].pct_change(5)
advanced['Volatility_14d'] = advanced['Return_1d'].rolling(14).std()
advanced['Volume_Ratio_7d'] = advanced['Volume'] / advanced['Volume'].rolling(7).mean() - 1
ADV_FEATURES = FEATURES + ['Return_7d','MA14_Ratio','Momentum_5d','Volatility_14d','Volume_Ratio_7d']
advanced[ADV_FEATURES] = advanced[ADV_FEATURES].replace([np.inf,-np.inf], np.nan)
advanced = advanced.dropna(subset=ADV_FEATURES).copy()
advanced['Next_Close'] = advanced['Close'].shift(-1)
advanced['Target'] = np.where(advanced['Next_Close'].notna(), (advanced['Next_Close'] > advanced['Close']).astype(int), np.nan)
latest_advanced = advanced[ADV_FEATURES].iloc[[-1]].copy()
advanced_data = advanced.dropna(subset=['Next_Close']).copy()
advanced_data['Target'] = advanced_data['Target'].astype(int)
# Same original test period, for fair comparison
cutoff = test.index.min()
adv_train = advanced_data.loc[advanced_data.index < cutoff]
adv_test = advanced_data.loc[advanced_data.index >= cutoff]
X_adv_train, y_adv_train = adv_train[ADV_FEATURES], adv_train['Target']
X_adv_test, y_adv_test = adv_test[ADV_FEATURES], adv_test['Target']
print('Advanced train/test:',len(adv_train),len(adv_test),'| Same test dates:',adv_test.index.equals(test.index))

## 12 — TimeSeriesSplit + GridSearchCV
نضبط النموذج على بيانات التدريب فقط. `gap=1` يبعد صفًا بين التدريب والتحقق لتقليل تسرب هدف اليوم التالي عند حدود الطيات. الاختبار النهائي يبقى مغلقًا.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import balanced_accuracy_score
cv = TimeSeriesSplit(n_splits=5, gap=1)
param_grid = {'n_estimators':[100,200], 'max_depth':[2,3,4], 'learning_rate':[0.01,0.05,0.1]}
search = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=1),
                      param_grid=param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=-1)
search.fit(X_adv_train, y_adv_train)
best_model = search.best_estimator_
print('Best CV balanced accuracy:',round(search.best_score_,3))
print('Best parameters:',search.best_params_)

## 13 — Fair Model Evaluation and Overfitting Check
نقارن على نفس تواريخ الاختبار: التوقع الدائم للارتفاع، تكرار اتجاه اليوم، النموذج الأساسي، والمحسن. الأداء فوق الصدفة لا يعني صلاحية التداول.

In [ ]:
from sklearn.metrics import precision_score, recall_score
adv_pred = best_model.predict(X_adv_test)
old_pred = model.predict(X_test)
always_up = np.ones(len(y_test),dtype=int)
same_today = (X_test['Return_1d'] > 0).astype(int)
rows=[]
for name,p in [('Always UP',always_up),('Same as today',same_today),('Original XGBoost',old_pred),('Tuned XGBoost',adv_pred)]:
    truth = y_adv_test if name=='Tuned XGBoost' else y_test
    rows.append({'Model':name,'Accuracy':accuracy_score(truth,p),'Balanced Accuracy':balanced_accuracy_score(truth,p),
                 'Precision UP':precision_score(truth,p,zero_division=0),'Recall UP':recall_score(truth,p,zero_division=0),
                 'F1 UP':f1_score(truth,p,zero_division=0)})
comparison = pd.DataFrame(rows)
display(comparison.round(3))
train_pred = best_model.predict(X_adv_train)
print('Train balanced accuracy:',round(balanced_accuracy_score(y_adv_train,train_pred),3))
print('CV balanced accuracy:',round(search.best_score_,3))
print('Test balanced accuracy:',round(balanced_accuracy_score(y_adv_test,adv_pred),3))
print('Tuned classification report:')
print(classification_report(y_adv_test,adv_pred,target_names=['No Increase','UP'],zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_adv_test,adv_pred,display_labels=['No Increase','UP'],cmap='Blues')
plt.title('Tuned XGBoost — Held-out Test');plt.show()

## 14 — Monitoring & MLOps Simulation
نقسم فترة الاختبار إلى فترات زمنية متتابعة ونراقب تغير الدقة. هذه مراقبة تاريخية وليست تشغيلًا مباشرًا أو إعادة تدريب تلقائيًا.

In [ ]:
monitor = pd.DataFrame({'Actual':y_adv_test,'Predicted':adv_pred},index=adv_test.index)
monitor['Quarter'] = monitor.index.to_period('Q').astype(str)
monitoring = monitor.groupby('Quarter').apply(lambda g: pd.Series({
    'Samples':len(g), 'Accuracy':accuracy_score(g['Actual'],g['Predicted']),
    'Balanced Accuracy':balanced_accuracy_score(g['Actual'],g['Predicted'])
}),include_groups=False).reset_index()
display(monitoring.round(3))
monitoring.plot(x='Quarter',y='Balanced Accuracy',marker='o',figsize=(10,4),title='Historical Monitoring: Balanced Accuracy')
plt.axhline(0.5,linestyle='--',color='gray');plt.ylim(0,1);plt.grid(True);plt.show()
print('Monitoring action: investigate sustained decline, inspect drift, retrain only after time-safe validation.')

## 15 — Next-session prediction and final export
توقع الاتجاه لليوم التالي لآخر جلسة متاحة؛ ليس توصية استثمارية. نحتفظ بالتاريخ الفعلي ونصدر ملفات التقييم والمراقبة.

## 16 — Export the trained model for GitHub chatbot
Run this after model training. Never commit model artifacts unless you intend to publish them.

In [ ]:
import json, joblib
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

adv_pred = best_model.predict(X_adv_test)
latest_prediction = int(best_model.predict(latest_advanced)[0])
latest_prob_up = float(best_model.predict_proba(latest_advanced)[0, 1])
joblib.dump(best_model, OUT / 'gold_xgboost.joblib')
metadata = {
    'ticker': 'GC=F',
    'last_date': str(latest_advanced.index[-1].date()),
    'last_close': float(advanced.loc[latest_advanced.index[-1], 'Close']),
    'features': ADV_FEATURES,
    'prediction': latest_prediction,
    'prob_up': latest_prob_up,
    'accuracy': float(accuracy_score(y_adv_test, adv_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_adv_test, adv_pred)),
    'f1_up': float(f1_score(y_adv_test, adv_pred, zero_division=0)),
    'test_samples': len(y_adv_test)
}
(OUT / 'model_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
print('Saved:', OUT / 'gold_xgboost.joblib', OUT / 'model_metadata.json')


## 17 — Gold AI Chatbot (Gradio)
Run after exporting model metadata. The assistant uses saved predictions; it is not a general-purpose LLM.

In [ ]:
!pip -q install gradio
import sys
sys.path.insert(0, '/content')
# In Colab upload app.py from the repository, or run the notebook's self-contained chatbot below.
import gradio as gr

def gold_chat(message, history):
    q = message.lower().strip()
    if any(x in q for x in ['اشتري','أشتري','ابيع','أبيع','شراء','بيع','buy','sell']):
        return 'النموذج تجريبي ولا يقدم توصيات شراء أو بيع. لا تعتمدي عليه وحده لاتخاذ قرار مالي.'
    if any(x in q for x in ['دقة','accuracy','f1','أداء','تقييم']):
        return f"Accuracy: {metadata['accuracy']:.1%}\n\nBalanced Accuracy: {metadata['balanced_accuracy']:.1%}\n\nF1 (UP): {metadata['f1_up']:.1%}"
    if any(x in q for x in ['احتمال','نسبة','probability','confidence']):
        return f"احتمال ارتفاع الذهب وفق النموذج: {metadata['prob_up']:.1%}. هذه درجة نموذج غير معايرة بالضرورة."
    if any(x in q for x in ['سعر','price','close']):
        return f"آخر إغلاق متاح لعقود الذهب الآجلة: ${metadata['last_close']:,.2f} للأونصة بتاريخ {metadata['last_date']}."
    if any(x in q for x in ['كيف','ليش','لماذا','xgboost','model','مؤشرات']):
        return 'يعتمد التنبؤ على XGBoost وخصائص مشتقة من الأسعار التاريخية والتذبذب والحجم. الاختبار زمني ولا يستخدم أسعار المستقبل في الخصائص.'
    if any(x in q for x in ['توقع','يرتفع','ينخفض','بكره','بكرة','غدا','غداً','اتجاه','predict','tomorrow']):
        direction = 'ارتفاع' if metadata['prediction'] == 1 else 'عدم ارتفاع'
        return f"التوقع: {direction} في جلسة التداول التالية لتاريخ {metadata['last_date']}. احتمال الارتفاع: {metadata['prob_up']:.1%}. هذا ليس توقعًا حيًا أو توصية استثمارية."
    return 'مرحبًا! اسأل عن توقع اتجاه الذهب أو احتمال الارتفاع أو آخر إغلاق أو دقة النموذج أو طريقة عمله.'

with gr.Blocks(theme=gr.themes.Soft(primary_hue='amber')) as demo:
    gr.Markdown('# Gold AI Assistant')
    gr.Markdown('Educational gold futures direction prediction | GC=F')
    gr.ChatInterface(fn=gold_chat, examples=['هل الذهب بيرتفع بكرة؟','كم احتمال الارتفاع؟','وش دقة النموذج؟','كم آخر سعر؟'])
demo.launch(share=True)
